In [55]:
import os
import json
import glob
import pandas as pd
import dask
import dask.bag as db
import dask.dataframe as dd
from datetime import datetime
from dask.distributed import Client

In [56]:
client=Client(n_workers=4)
print(client.dashboard_link)

d:\analizadaych\.venv\Lib\site-packages\distributed\node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 57905 instead
  warnings.warn(


http://127.0.0.1:57905/status


### Zadanie 1

In [57]:
with open("Apache_2k.log",encoding="utf8") as f:
    base_lines=[line.strip() for line in f if line.strip()]

#### Powielenie logów na 500000 linii

In [58]:
lines=base_lines*250
print(len(lines))

500000


In [59]:
def parse(inp: str):
    record = {}
    
    date_start = inp.find('[') + 1
    date_end = inp.find(']')
    date_s = slice(date_start, date_end)

    level_start = inp.find('[', date_end) + 1
    level_end = inp.find(']', level_start)
    level_s = slice(level_start, level_end)

    client_start = inp.find('[client ', level_end)
    client_end = inp.find(']', client_start)

    record["date"] = inp[date_s]    
    record["level"] = inp[level_s]
    record["client"] = "" if client_start == -1 else inp[client_start + 8: client_end]
    record["message"] = inp[client_end + 2:] if record["client"] else inp[level_end + 2:]
    
    return record

def convert_date(rec):
    rec["date"] = datetime.strptime(rec["date"], "%a %b %d %H:%M:%S %Y")

    return rec

start=datetime.now()
output = []
for line in lines:
    record = convert_date(parse(line))
    output.append(list(record.values()))

df_seq = pd.DataFrame(output, columns=["date", "level", "client", "message"])

df_seq.to_parquet("Apache_2k_log.parquet")


#### Czas obliczeń

In [60]:
time_seq=datetime.now()-start
print("Czas",time_seq)

Czas 0:00:04.199894


In [61]:
df_seq.head()

,date,level,client,message
0,2005-12-04 04:47:44,notice,,workerEnv.init() ok /etc/httpd/conf/workers2.p...
1,2005-12-04 04:47:44,error,,mod_jk child workerEnv in error state 6
2,2005-12-04 04:51:08,notice,,jk2_init() Found child 6725 in scoreboard slot 10
3,2005-12-04 04:51:09,notice,,jk2_init() Found child 6726 in scoreboard slot 8
4,2005-12-04 04:51:09,notice,,jk2_init() Found child 6728 in scoreboard slot 6


#### Podział na paczki po 50000 linii

In [62]:
def parse_chunk(chunk):
    output=[]
    for line in chunk:
        record = convert_date(parse(line))
        output.append(list(record.values()))
    return pd.DataFrame(output,columns=["date","level","client","message"])

chunk_size=50_000
chunks=[lines[i:i+chunk_size]for i in range(0,len(lines),chunk_size)]
print("Paczki",len(chunks))

Paczki 10


In [63]:
start=datetime.now()
parts=[dask.delayed(parse_chunk)(chunk)for chunk in chunks]
meta=parse_chunk(lines[:1]).iloc[:0]
ddf_apache=dd.from_delayed(parts,meta=meta)
ddf_apache.to_parquet("Apache_2k_log_dask",write_index=False)

timee=datetime.now()-start
print("Równoległość",timee)


Równoległość 0:00:03.162533


#### Porównanie czasów

In [64]:
print("sekwencyjne:",time_seq)
print("równolegle",timee)

sekwencyjne: 0:00:04.199894
równolegle 0:00:03.162533


### Zadanie 2

In [65]:
DATAPATH='data/people'
os.makedirs(DATAPATH,exist_ok=True)
people=dask.datasets.make_people(npartitions=100,records_per_partition=1000)
people.map(json.dumps).to_textfiles(os.path.join(DATAPATH,'*.json'))
print("Liczba",len(glob.glob(os.path.join(DATAPATH,'*.json'))))


Liczba 110


#### Wczytanie danych jako Dask Bag

In [66]:
bag=db.read_text(os.path.join(DATAPATH,'*.json')).map(json.loads)
bag.take(1)

({'age': 64,
  'name': ['Benton', 'Campbell'],
  'occupation': 'Security Guard',
  'telephone': '+17640810499',
  'address': {'address': '1032 Brompton Boulevard', 'city': 'Chickasha'},
  'credit-card': {'number': '3402 646121 27564', 'expiration-date': '03/21'}},)

#### Czy karta wygasła

In [67]:
now=datetime.now()
def is_expired(record):
    month,year=record['credit-card']['expiration-date'].split('/')
    return (2000 +int(year),int(month))<(now.year,now.month)

is_expired(bag.take(1)[0])

True

#### Filtr wygasłych kart

In [68]:
start=datetime.now()
expired=bag.filter(is_expired).repartition(10)
expired.map(json.dumps).to_textfiles(os.path.join(DATAPATH,'expired_*.json'))
print("Czas",datetime.now()-start)
print("Liczba wygaśnięć",len(glob.glob(os.path.join(DATAPATH,'expired_*.json'))))

Czas 0:00:02.846655
Liczba wygaśnięć 10


#### Ile rekordów wygaslo

In [69]:
expired_check=db.read_text(os.path.join(DATAPATH,'expired_*.json')).map(json.loads)
print("Wygasłe:", expired_check.count().compute(),"z",bag.count().compute())

Wygasłe: 200000 z 300000


### Zadanie 3

#### Wybór dorosłych

In [70]:
adults=bag.filter(lambda record:record['age']>=18)
adults.take(1)

({'age': 64,
  'name': ['Benton', 'Campbell'],
  'occupation': 'Security Guard',
  'telephone': '+17640810499',
  'address': {'address': '1032 Brompton Boulevard', 'city': 'Chickasha'},
  'credit-card': {'number': '3402 646121 27564', 'expiration-date': '03/21'}},)

#### Spłaszczenie słownika

In [71]:
def flat(record):
    return {'age':record['age'],'first_name':record['name'][0],'last_name':record['name'][1],'occupation':record['occupation'],'street':record['address']['address'],'city':record['address']['city'],'card_number':record['credit-card']['number'],'card_expiration':record['credit-card']['expiration-date'],}
adults_flat=adults.map(flat)
adults_flat.take(1)

({'age': 64,
  'first_name': 'Benton',
  'last_name': 'Campbell',
  'occupation': 'Security Guard',
  'street': '1032 Brompton Boulevard',
  'city': 'Chickasha',
  'card_number': '3402 646121 27564',
  'card_expiration': '03/21'},)

#### Zmiana na Daskdataframe

In [72]:
ddf_adults=adults_flat.to_dataframe()
print("Liczba partycji",ddf_adults.npartitions)
ddf_adults.head()

Liczba partycji 110


,age,first_name,last_name,occupation,street,city,card_number,card_expiration
0,64,Benton,Campbell,Security Guard,1032 Brompton Boulevard,Chickasha,3402 646121 27564,03/21
1,91,Sari,Joyce,Travel Consultant,1329 Delano Canyon,Clinton,5396 3246 2370 9214,11/20
2,54,Franklin,Lowery,Glass Worker,1284 Cumberland Trail,College Station,5382 1277 8316 2553,10/16
3,37,Talitha,Macdonald,Instrument Maker,851 Seal Cove Alley,Pharr,3423 420759 07785,10/19
4,51,Gilbert,Everett,Gilder,1289 Dwight Trace,Fountain Hills,5579 2749 8351 8896,10/17


#### Zapis do parquet

In [73]:
start=datetime.now()
ddf_adults.repartition(npartitions=1).to_parquet("data/adults_parquet",write_index=False)
print("Czas",datetime.now()-start)
print(os.listdir("data/adults_parquet"))

Czas 0:00:02.413859
['part.0.parquet']
